In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT_DIR = Path.cwd().parent
DATA_DIR = ROOT_DIR / "data" / "raw"
PROCESSED_DIR = ROOT_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

files = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

dfs = {
    name: pd.read_csv(DATA_DIR / filename)
    for name, filename in files.items()
}

In [2]:
def clean_column_names(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )
    return df


for name in dfs:
    dfs[name] = clean_column_names(dfs[name])

In [3]:
for name, df in dfs.items():
    print(name, df.columns.tolist())

customers ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
geolocation ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']
order_items ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
order_payments ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']
order_reviews ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']
orders ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
products ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 

In [4]:
def data_quality_report(df):
    report = pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str),
        "row_count": len(df),
        "missing_count": df.isna().sum().values,
        "missing_pct": (
            df.isna().mean().values * 100
        ).round(2),
        "unique_count": df.nunique().values,
        "duplicate_count": [
            df[col].duplicated().sum()
            for col in df.columns
        ]
    })

    return report

In [17]:
orders_quality = data_quality_report(
    dfs["orders"]
)

orders_quality

,column,dtype,row_count,missing_count,missing_pct,unique_count,duplicate_count
order_id,order_id,object,99441,0,0.00,99441,0
customer_id,customer_id,object,99441,0,0.00,99441,0
order_status,order_status,object,99441,0,0.00,8,99433
order_purchase_timestamp,order_purchase_timestamp,object,99441,0,0.00,98875,566
order_approved_at,order_approved_at,object,99441,160,0.16,90733,8707
order_delivered_carrier_date,order_delivered_carrier_date,object,99441,1783,1.79,81018,18422
order_delivered_customer_date,order_delivered_customer_date,object,99441,2965,2.98,95664,3776
order_estimated_delivery_date,order_estimated_delivery_date,object,99441,0,0.00,459,98982


In [19]:
for name, df in dfs.items():
    print(f"\n{name.upper()}")
    display(data_quality_report(df))


CUSTOMERS


,column,dtype,row_count,missing_count,missing_pct,unique_count,duplicate_count
customer_id,customer_id,object,99441,0,0.0,99441,0
customer_unique_id,customer_unique_id,object,99441,0,0.0,96096,3345
customer_zip_code_prefix,customer_zip_code_prefix,int64,99441,0,0.0,14994,84447
customer_city,customer_city,object,99441,0,0.0,4119,95322
customer_state,customer_state,object,99441,0,0.0,27,99414



GEOLOCATION


,column,dtype,row_count,missing_count,missing_pct,unique_count,duplicate_count
geolocation_zip_code_prefix,geolocation_zip_code_prefix,int64,1000163,0,0.0,19015,981148
geolocation_lat,geolocation_lat,float64,1000163,0,0.0,717360,282803
geolocation_lng,geolocation_lng,float64,1000163,0,0.0,717613,282550
geolocation_city,geolocation_city,object,1000163,0,0.0,8011,992152
geolocation_state,geolocation_state,object,1000163,0,0.0,27,1000136



ORDER_ITEMS


,column,dtype,row_count,missing_count,missing_pct,unique_count,duplicate_count
order_id,order_id,object,112650,0,0.0,98666,13984
order_item_id,order_item_id,int64,112650,0,0.0,21,112629
product_id,product_id,object,112650,0,0.0,32951,79699
seller_id,seller_id,object,112650,0,0.0,3095,109555
shipping_limit_date,shipping_limit_date,object,112650,0,0.0,93318,19332
price,price,float64,112650,0,0.0,5968,106682
freight_value,freight_value,float64,112650,0,0.0,6999,105651


... (8 outputs left out)


SELLERS


,column,dtype,row_count,missing_count,missing_pct,unique_count,duplicate_count
seller_id,seller_id,object,3095,0,0.0,3095,0
seller_zip_code_prefix,seller_zip_code_prefix,int64,3095,0,0.0,2246,849
seller_city,seller_city,object,3095,0,0.0,611,2484
seller_state,seller_state,object,3095,0,0.0,23,3072



CATEGORY_TRANSLATION


,column,dtype,row_count,missing_count,missing_pct,unique_count,duplicate_count
product_category_name,product_category_name,object,71,0,0.0,71,0
product_category_name_english,product_category_name_english,object,71,0,0.0,71,0


In [20]:
orders = dfs["orders"].copy()

orders.isna().sum().sort_values(ascending=False)

order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
order_id                            0
order_purchase_timestamp            0
order_status                        0
customer_id                         0
order_estimated_delivery_date       0
dtype: int64

In [21]:
orders.groupby("order_status").agg(
    orders=("order_id", "count"),
    delivered_date_missing=(
        "order_delivered_customer_date",
        lambda x: x.isna().sum()
    ),
    carrier_date_missing=(
        "order_delivered_carrier_date",
        lambda x: x.isna().sum()
    )
)

,orders,delivered_date_missing,carrier_date_missing
order_status,,,
approved,2,2,2
canceled,625,619,550
created,5,5,5
delivered,96478,8,2
invoiced,314,314,314
processing,301,301,301
shipped,1107,1107,0
unavailable,609,609,609


In [22]:
orders[
    orders["order_delivered_customer_date"]
    < orders["order_purchase_timestamp"]
][[
    "order_id",
    "order_purchase_timestamp",
    "order_delivered_customer_date"
]]

,order_id,order_purchase_timestamp,order_delivered_customer_date


In [26]:
orders[
    orders["order_delivered_customer_date"]
    < orders["order_delivered_carrier_date"]
][[
    "order_id",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]]

,order_id,order_delivered_carrier_date,order_delivered_customer_date
6437,a1abeb653a4d4cd1e142ccb8c82cd069,2017-07-28 16:57:58,2017-07-25 19:32:56
9553,383aa8b2724fe452d9ccd9934a8c628b,2017-07-07 17:22:41,2017-07-06 14:27:51
13487,cb1134f9010d242e9515ad1c78ec0c39,2017-07-20 19:22:02,2017-07-19 14:13:28
14474,dceb62e8fa94b46006c9554fed743df0,2017-08-01 18:23:30,2017-07-26 18:09:10
19268,5f9d46795c3126674e52becb3a1a517f,2017-07-20 23:03:42,2017-07-20 18:52:41
21338,8c78d01de3a9009e23d6877a7cc9be20,2016-10-26 11:41:53,2016-10-25 17:51:46
22520,b27af682321527a6349f1761eb3f360c,2017-06-27 14:51:54,2017-06-26 15:45:35
25393,1cc3ae63caffff2d6c3ee3e78e074acf,2017-08-10 18:28:56,2017-08-10 18:05:38
25646,e37f11cae9985ca58f0b56f268720537,2017-08-01 18:17:47,2017-07-31 17:49:56
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,2017-08-09 18:18:43,2017-08-01 21:13:01


In [24]:
order_items = dfs["order_items"].copy()

order_items[
    order_items["price"] <= 0
]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


In [27]:
order_items[
    order_items["freight_value"] < 0
]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


In [28]:
products = dfs["products"].copy()

products.isna().sum().sort_values(
    ascending=False
)

product_category_name         610
product_description_lenght    610
product_name_lenght           610
product_photos_qty            610
product_weight_g                2
product_height_cm               2
product_length_cm               2
product_width_cm                2
product_id                      0
dtype: int64

In [29]:
products.describe()

,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32341.000000,32341.000000,32341.000000,32949.000000,32949.000000,32949.000000,32949.000000
mean,48.476949,771.495285,2.188986,2276.472488,30.815078,16.937661,23.196728
std,10.245741,635.115225,1.736766,4282.038731,16.914458,13.637554,12.079047
min,5.000000,4.000000,1.000000,0.000000,7.000000,2.000000,6.000000
25%,42.000000,339.000000,1.000000,300.000000,18.000000,8.000000,15.000000
50%,51.000000,595.000000,1.000000,700.000000,25.000000,13.000000,20.000000
75%,57.000000,972.000000,3.000000,1900.000000,38.000000,21.000000,30.000000
max,76.000000,3992.000000,20.000000,40425.000000,105.000000,105.000000,118.000000


In [30]:
numeric_product_cols = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for col in numeric_product_cols:
    print(
        col,
        "zero =", (products[col] == 0).sum(),
        "negative =", (products[col] < 0).sum()
    )

product_weight_g zero = 4 negative = 0
product_length_cm zero = 0 negative = 0
product_height_cm zero = 0 negative = 0
product_width_cm zero = 0 negative = 0


In [31]:
customers = dfs["customers"].copy()

customers["customer_id"].duplicated().sum()

np.int64(0)

In [32]:
products["product_id"].duplicated().sum()

np.int64(0)

In [33]:
sellers = dfs["sellers"].copy()

sellers["seller_id"].duplicated().sum()

np.int64(0)

In [34]:
orders["order_id"].duplicated().sum()

np.int64(0)

In [35]:
dim_customer = customers[[
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state"
]].copy()

In [36]:
dim_customer = dim_customer.rename(columns={
    "customer_zip_code_prefix": "customer_zip_code"
})

In [37]:
dim_seller = sellers[[
    "seller_id",
    "seller_zip_code_prefix",
    "seller_city",
    "seller_state"
]].copy()

dim_seller = dim_seller.rename(columns={
    "seller_zip_code_prefix": "seller_zip_code"
})

In [38]:
category_translation = dfs[
    "category_translation"
].copy()

products = products.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

In [39]:
products[
    [
        "product_id",
        "product_category_name",
        "product_category_name_english"
    ]
].head()

,product_id,product_category_name,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,housewares


In [40]:
dim_product = products[[
    "product_id",
    "product_category_name",
    "product_category_name_english",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]].copy()

In [42]:
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

date_range = pd.date_range(
    start=orders["order_purchase_timestamp"].min().date(),
    end=orders["order_purchase_timestamp"].max().date(),
    freq="D"
)

dim_date = pd.DataFrame({
    "date": date_range
})

In [43]:
dim_date["year"] = dim_date["date"].dt.year
dim_date["quarter"] = (
    "Q" + dim_date["date"].dt.quarter.astype(str)
)
dim_date["month"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.month_name()
dim_date["year_month"] = (
    dim_date["date"].dt.to_period("M").astype(str)
)
dim_date["week"] = dim_date["date"].dt.isocalendar().week
dim_date["day"] = dim_date["date"].dt.day
dim_date["day_name"] = dim_date["date"].dt.day_name()

In [44]:
fact_order = orders[[
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]].copy()

In [49]:
date_columns = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    fact_order[col] = pd.to_datetime(
        fact_order[col],
        errors="coerce"
    )

fact_order["delivery_days"] = (
    fact_order["order_delivered_customer_date"]
    - fact_order["order_purchase_timestamp"]
).dt.total_seconds() / 86400

In [50]:
print(fact_order[[
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "delivery_days"
]].head())


  order_purchase_timestamp order_delivered_customer_date  delivery_days
0      2017-10-02 10:56:33           2017-10-10 21:25:13       8.436574
1      2018-07-24 20:41:37           2018-08-07 15:27:45      13.782037
2      2018-08-08 08:38:49           2018-08-17 18:06:29       9.394213
3      2017-11-18 19:28:06           2017-12-02 00:28:42      13.208750
4      2018-02-13 21:18:39           2018-02-16 18:17:02       2.873877


In [51]:
fact_order["estimated_delay_days"] = (
    fact_order["order_delivered_customer_date"]
    - fact_order["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

In [52]:
fact_order["is_late"] = (
    fact_order["estimated_delay_days"] > 0
).astype(int)

In [53]:
fact_order_item = order_items[[
    "order_id",
    "order_item_id",
    "product_id",
    "seller_id",
    "shipping_limit_date",
    "price",
    "freight_value"
]].copy()

In [54]:
fact_order_item["gross_value"] = (
    fact_order_item["price"]
    + fact_order_item["freight_value"]
)

In [55]:
dim_customer.to_csv(
    PROCESSED_DIR / "dim_customer.csv",
    index=False
)

dim_seller.to_csv(
    PROCESSED_DIR / "dim_seller.csv",
    index=False
)

dim_product.to_csv(
    PROCESSED_DIR / "dim_product.csv",
    index=False
)

dim_date.to_csv(
    PROCESSED_DIR / "dim_date.csv",
    index=False
)

fact_order.to_csv(
    PROCESSED_DIR / "fact_order.csv",
    index=False
)

fact_order_item.to_csv(
    PROCESSED_DIR / "fact_order_item.csv",
    index=False
)

In [56]:
print("dim_customer :", len(dim_customer))
print("dim_seller   :", len(dim_seller))
print("dim_product  :", len(dim_product))
print("dim_date     :", len(dim_date))
print("fact_order   :", len(fact_order))
print("fact_item    :", len(fact_order_item))

dim_customer : 99441
dim_seller   : 3095
dim_product  : 32951
dim_date     : 774
fact_order   : 99441
fact_item    : 112650


In [57]:
assert dim_customer["customer_id"].is_unique
assert dim_seller["seller_id"].is_unique
assert dim_product["product_id"].is_unique
assert fact_order["order_id"].is_unique

In [58]:
fact_order["order_purchase_date"] = (
    fact_order["order_purchase_timestamp"].dt.date
)

fact_order["order_purchase_date"] = pd.to_datetime(
    fact_order["order_purchase_date"]
)

In [59]:
fact_order["order_delivered_date"] = (
    fact_order["order_delivered_customer_date"].dt.date
)

fact_order["order_delivered_date"] = pd.to_datetime(
    fact_order["order_delivered_date"]
)

In [60]:
fact_order.to_csv(
    PROCESSED_DIR / "fact_order.csv",
    index=False
)